# AI Essay Detector - Step 1: Data Cleaning

This notebook cleans the raw text dataset, stripping trailing whitespace and filtering out truncated essays containing fewer than 100 words.


### Kaggle Dataset Presence Check & Downloader
This block checks if the dataset `train_v4_drcat_01.csv` is present locally. If not, it sets up Kaggle, downloads the dataset zip, extracts it, and performs cleanup.

In [ ]:
import os

DATASET_FILE_NAME = "train_v4_drcat_01.csv"
KAGGLE_ZIP_FILE_NAME = "daigt-v4-train-dataset.zip"

# Check if the dataset file already exists locally
if os.path.exists(DATASET_FILE_NAME):
    print(f"'{DATASET_FILE_NAME}' found locally. Skipping Kaggle download.")
else:
    print(f"'{DATASET_FILE_NAME}' not found locally. Attempting to download from Kaggle...")

    # 1. Install the Kaggle library
    !pip install kaggle --quiet

    # Ensure the .kaggle directory exists (redundant if previous cell ran, but safe)
    !mkdir -p ~/.kaggle

    # The kaggle.json file should have been created by the previous cell
    # To prevent 'kaggle.json' from being publicly readable
    !chmod 600 ~/.kaggle/kaggle.json

    print("Kaggle API setup should be complete (kaggle.json created by previous cell).")

    # 3. Download the dataset
    dataset_name = "thedrcat/daigt-v4-train-dataset"
    !kaggle datasets download -d {dataset_name}

    # 4. Unzip the downloaded file
    !unzip -o {KAGGLE_ZIP_FILE_NAME} -d .

    # 5. Clean up the zip file (optional)
    !rm {KAGGLE_ZIP_FILE_NAME}

    if os.path.exists(DATASET_FILE_NAME):
        print(f"\nDataset '{dataset_name}' downloaded and unzipped successfully.")
        print(f"You should now see '{DATASET_FILE_NAME}' in your current directory.")
    else:
        raise FileNotFoundError(f"Failed to download and extract '{DATASET_FILE_NAME}' from Kaggle.")


### Working Directory & Path Initialization
Since this notebook runs inside the `notebooks/` directory, we shift the kernel's active path to the project root (`..`) so that relative data paths map correctly. We also add `src/` to the python import path so that custom helper modules like `evaluate` can be imported.

In [ ]:
import os
import sys

# If running from notebooks/ directory, switch to the project root
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

# Ensure the src/ directory is in the import path so evaluate can be found
sys.path.append(os.path.abspath('src'))

print(f"Current Working Directory set to: {os.getcwd()}")
print("Import path appended. You can now import evaluation utilities successfully.")


### 1. Import Required Libraries
We import `pandas` and `numpy` for data manipulation, and the standard library `os` to verify files and resolve paths.


In [ ]:
import os
import pandas as pd
import numpy as np


### 2. Dataset Cleaning Function
This core cleaning function performs several validation and preprocessing steps:
- **File Check**: Raises `FileNotFoundError` if the raw dataset is missing.
- **Whitespace Stripping**: Removes trailing and leading whitespace to clean formatting issues.
- **Word Count**: Calculates the word count of each essay.
- **Length-Based Filtering**: Removes corrupted essays (fewer than 100 words) and prints sample filtered entries for logging.
- **Distribution Checking**: Logs the label proportions (AI vs. Human) of the clean data.
- **Export**: Exports the cleaned DataFrame back to disk.


In [ ]:
def clean_dataset(input_path, output_path):
    print("=== Loading Raw Dataset ===")
    if not os.path.exists(input_path):
        raise FileNotFoundError(f"Input file not found at {input_path}")
        
    df = pd.read_csv(input_path)
    initial_shape = df.shape
    print(f"Loaded {initial_shape[0]} rows and {initial_shape[1]} columns.")
    
    print("\n=== Preprocessing & Data Cleaning ===")
    
    # 1. Strip leading and trailing whitespace from text
    print("Normalizing whitespaces...")
    df['text'] = df['text'].astype(str).str.strip()
    
    # 2. Calculate word count for filtering
    print("Calculating word counts...")
    df['word_count'] = df['text'].apply(lambda x: len(x.split()))
    
    # 3. Identify and filter corrupted/ultra-short texts
    # We filter out text with fewer than 100 words as these are prompt titles, truncated lines, or gibberish.
    min_word_limit = 100
    corrupted_mask = df['word_count'] < min_word_limit
    num_corrupted = corrupted_mask.sum()
    
    print(f"Found {num_corrupted} rows with word count < {min_word_limit} (corrupted/truncated essays).")
    
    # Let's log some examples of filtered texts for transparency
    if num_corrupted > 0:
        print("\nExamples of filtered short/corrupted texts:")
        sample_corrupted = df[corrupted_mask].sort_values(by='word_count').head(5)
        for idx, row in sample_corrupted.iterrows():
            print(f" - [Label: {row['label']}, Source: {row['source']}, Words: {row['word_count']}] Text: {repr(row['text'][:150])}...")
            
    # Remove corrupted rows
    df_clean = df[~corrupted_mask].copy()
    
    final_shape = df_clean.shape
    rows_removed = initial_shape[0] - final_shape[0]
    
    print(f"\nFiltered out {rows_removed} rows. Cleaned dataset has {final_shape[0]} rows.")
    
    # 4. Check label distribution in the cleaned dataset
    print("\n=== Cleaned Label Distribution ===")
    label_counts = df_clean['label'].value_counts()
    for label, count in label_counts.items():
        percentage = (count / final_shape[0]) * 100
        label_name = "AI-Generated (1)" if label == 1 else "Human-Written (0)"
        print(f" - {label_name}: {count} ({percentage:.2f}%)")
        
    # 5. Save the cleaned dataset
    print("\n=== Saving Cleaned Dataset ===")
    df_clean.to_csv(output_path, index=False)
    print(f"Cleaned dataset successfully saved to: {output_path}")
    print(f"File size: {os.path.getsize(output_path) / (1024 * 1024):.2f} MB")


### 3. Execution Entry Point
Defines the default file names and triggers the dataset cleaning pipeline.


In [ ]:
if __name__ == "__main__":
    INPUT_FILE = "train_v4_drcat_01.csv"
    OUTPUT_FILE = "train_v4_drcat_01_cleaned.csv"
    
    clean_dataset(INPUT_FILE, OUTPUT_FILE)
